# Clipping

In [25]:
#función para reutilizar cada q hay CT  Valores según recomendación
def preprocess_ct_scan(scan_path, hu_min=-1000, hu_max=400):
    sitk_image = sitk.ReadImage(scan_path)
    hu_array = sitk.GetArrayFromImage(sitk_image)
    #pa recortar valores que se pasan osea si hay -3000 va ser el -1000
    clipped = np.clip(hu_array, hu_min, hu_max)
    #Aparte de los valores recomendados se tiene q hacer ma chiquito osea 1 o 0 
    normalized = (clipped - hu_min) / (hu_max - hu_min)
    #tmb el float64 a 32 para consumir menos 
    normalized = normalized.astype(np.float32)
    return normalized, hu_array

In [77]:
#Las originales
subset_folder = "data/subset9"
#Se crea esta donde se va guardar las procesadas
output_dir = "processed_subset9"
os.makedirs(output_dir, exist_ok=True)
#Aqui recorre el subset0
for file in os.listdir(subset_folder):
    #Solo los que terminen en .mhd
    if file.endswith(".mhd"):
        #Construye la ruta 
        path = os.path.join(subset_folder, file)
        #se le quita .mhd
        seriesuid = file.replace(".mhd", "")
        #Crea nombre del archivo
        output_file = os.path.join(output_dir, f"{seriesuid}.npy")
        try:
            scan_norm, _ = preprocess_ct_scan(path)
            np.save(output_file, scan_norm)
            print(f"Guardado: {seriesuid}")
        except Exception as e:
            print(f"Error: {e}")

Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.102681962408431413578140925249
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.109882169963817627559804568094
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.112767175295249119452142211437
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.114914167428485563471327801935
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.121108220866971173712229588402
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.121805476976020513950614465787
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.124656777236468248920498636247
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.124822907934319930841506266464
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.129650136453746261130135157590
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.134519406153127654901640638633
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.138674679709964033277400089532
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.138813197521718693188313387015
Guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.139577698050713461261415990027
Guardado: 1.3.6.1.4.1.145

# Extraer patches  ESTE NO VALE me dio lapsus brutus JAJAAJ

In [78]:
#Para guardar los patches
patch_dir = "patches_subset9"
os.makedirs(patch_dir, exist_ok=True)

In [18]:
#tompgrafia completa,centro del nodulo,tamaño del patch
def extract_patch(scan_3d, center_voxel, patch_size=(64,64,64)):
    z, y, x = center_voxel
    dz, dy, dx = patch_size
    half_z = dz // 2
    half_y = dy // 2
    half_x = dx // 2
    #Para evitar numeros negativos
    z_min = max(0, z - half_z)
    #Para evitar salirme del scan
    z_max = min(scan_3d.shape[0], z + half_z)
    y_min = max(0, y - half_y)
    y_max = min(scan_3d.shape[1], y + half_y)
    x_min = max(0, x - half_x)
    x_max = min(scan_3d.shape[2], x + half_x)
    #Cortar el cubito
    patch = scan_3d[
        z_min:z_max,
        y_min:y_max,
        x_min:x_max
    ]
    return patch

In [79]:
subset_folder = "data/subset9"
count = 0
for idx, row in annotations.iterrows():
    seriesuid = row['seriesuid']
    coordX = row['coordX']
    coordY = row['coordY']
    coordZ = row['coordZ']
    found_path = None
    #Para ver si ese nodilo es del subset0
    for file in os.listdir(subset_folder):
        if file.startswith(seriesuid) and file.endswith(".mhd"):
            found_path = os.path.join(subset_folder, file)
            break
    if not found_path:
        continue
    try:
        sitk_img = sitk.ReadImage(found_path)
        #Inicio tomografia
        origin = sitk_img.GetOrigin()
        #Tamaño voxel
        spacing = sitk_img.GetSpacing()
        #Coordenada real a voxel
        voxelX = int((coordX - origin[0]) / spacing[0])
        voxelY = int((coordY - origin[1]) / spacing[1])
        voxelZ = int((coordZ - origin[2]) / spacing[2])
        norm_path = os.path.join(
            output_dir,
            f"{seriesuid}.npy"
        )
        if not os.path.exists(norm_path):
            continue
        scan_norm = np.load(norm_path)
        #Cortamo el patch
        patch = extract_patch(
            scan_norm,
            (voxelZ, voxelY, voxelX),
            patch_size=(64,64,64)
        )
        #Verificamo si efectivamente es de 64
        if patch.shape == (64,64,64):
            save_name = f"{seriesuid}_{idx}.npy"
            save_path = os.path.join(
                patch_dir,
                save_name
            )
            np.save(save_path, patch)
            count += 1
            print(f"Patch guardado: {save_name}")
    except Exception as e:
        print(f"Error: {e}")
print(f"Total patches: {count}")

Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.112767175295249119452142211437_37.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.112767175295249119452142211437_39.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.112767175295249119452142211437_40.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.112767175295249119452142211437_43.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.114914167428485563471327801935_54.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.114914167428485563471327801935_55.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.121108220866971173712229588402_70.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.124822907934319930841506266464_90.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.124822907934319930841506266464_91.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.124822907934319930841506266464_92.npy
Patch guardado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.124822907934319930841506266464_93.npy
Patch guardado: 1.3.6.1.4.1.1451

In [80]:
subset9_seriesuid = []
for file in os.listdir(subset_folder):
    if file.endswith(".mhd"):
        seriesuid = file.replace(".mhd", "")
        subset9_seriesuid.append(seriesuid)
positive_seriesuid = set(annotations['seriesuid'].unique())
negative_subset9 = []
for s in subset9_seriesuid:
    if s not in positive_seriesuid:
        negative_subset9.append(s)
print("Scans negativos subset7:", len(negative_subset9))

Scans negativos subset7: 29


In [81]:
negative_dir = "negative_patches_subset9"
os.makedirs(negative_dir, exist_ok=True)
count_neg = 0
patches_per_scan = 5
for seriesuid in negative_subset9:
    norm_path = os.path.join(output_dir,f"{seriesuid}.npy")
    if not os.path.exists(norm_path):continue
    scan = np.load(norm_path)
    z, y, x = scan.shape
    for i in range(patches_per_scan):
        cz = random.randint(32, z-32)
        cy = random.randint(32, y-32)
        cx = random.randint(32, x-32)
        patch = extract_patch(scan,(cz, cy, cx),patch_size=(64,64,64))
        if patch.shape == (64,64,64):
            save_path = os.path.join(negative_dir,f"negative_{count_neg}.npy")
            np.save(save_path, patch)
            count_neg += 1
            print(f"Negativo guardado {count_neg}")

Negativo guardado 1
Negativo guardado 2
Negativo guardado 3
Negativo guardado 4
Negativo guardado 5
Negativo guardado 6
Negativo guardado 7
Negativo guardado 8
Negativo guardado 9
Negativo guardado 10
Negativo guardado 11
Negativo guardado 12
Negativo guardado 13
Negativo guardado 14
Negativo guardado 15
Negativo guardado 16
Negativo guardado 17
Negativo guardado 18
Negativo guardado 19
Negativo guardado 20
Negativo guardado 21
Negativo guardado 22
Negativo guardado 23
Negativo guardado 24
Negativo guardado 25
Negativo guardado 26
Negativo guardado 27
Negativo guardado 28
Negativo guardado 29
Negativo guardado 30
Negativo guardado 31
Negativo guardado 32
Negativo guardado 33
Negativo guardado 34
Negativo guardado 35
Negativo guardado 36
Negativo guardado 37
Negativo guardado 38
Negativo guardado 39
Negativo guardado 40
Negativo guardado 41
Negativo guardado 42
Negativo guardado 43
Negativo guardado 44
Negativo guardado 45
Negativo guardado 46
Negativo guardado 47
Negativo guardado 48
N

# Extraer patches

In [2]:
candidates = pd.read_csv("data/candidates.csv")
print(candidates.head())

                                           seriesuid  coordX  coordY  coordZ  \
0  1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...  -56.08  -67.85 -311.92   
1  1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...   53.21 -244.41 -245.17   
2  1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...  103.66 -121.80 -286.62   
3  1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...  -33.66  -72.75 -308.41   
4  1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...  -32.25  -85.36 -362.51   

   class  
0      0  
1      0  
2      0  
3      0  
4      0  


In [4]:
patch_dir = "patches_candidates"
os.makedirs(patch_dir, exist_ok=True)

In [18]:
def extract_patch(scan_3d, center_voxel, patch_size=(64,64,64)):
    z, y, x = center_voxel
    dz, dy, dx = patch_size
    half_z = dz // 2
    half_y = dy // 2
    half_x = dx // 2
    z_min = max(0, z - half_z)
    z_max = min(scan_3d.shape[0], z + half_z)
    y_min = max(0, y - half_y)
    y_max = min(scan_3d.shape[1], y + half_y)
    x_min = max(0, x - half_x)
    x_max = min(scan_3d.shape[2], x + half_x)
    patch = scan_3d[
        z_min:z_max,
        y_min:y_max,
        x_min:x_max
    ]
    return patch

In [14]:
processed_paths = {}
for subset in range(10):
    folder = f"processed_subset{subset}"
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            seriesuid = file.replace(".npy", "")
            processed_paths[seriesuid] = os.path.join(folder, file)
print("Scans encontrados:", len(processed_paths))

Scans encontrados: 782


In [16]:
metadata_cache = {}
for subset in range(10):
    folder = f"data/subset{subset}"
    for file in os.listdir(folder):
        if file.endswith(".mhd"):
            path = os.path.join(folder, file)
            seriesuid = file.replace(".mhd", "")
            try:
                sitk_img = sitk.ReadImage(path)
                metadata_cache[seriesuid] = {
                    "origin": sitk_img.GetOrigin(),
                    "spacing": sitk_img.GetSpacing()
                }
            except Exception as e:
                print("Error metadata:", e)

In [10]:
count = 0
for idx, row in candidates.iterrows():
    seriesuid = row['seriesuid']
    coordX = row['coordX']
    coordY = row['coordY']
    coordZ = row['coordZ']
    label = row['class']
    norm_path = processed_paths.get(seriesuid)
    if not norm_path:
        continue
    try:
        # cargar scan preprocessado
        scan_norm = np.load(norm_path)
        # obtener metadata desde RAM
        metadata = metadata_cache.get(seriesuid)
        if metadata is None:continue
        origin = metadata["origin"]
        spacing = metadata["spacing"]
        # coordenadas reales -> voxel
        voxelX = int((coordX - origin[0]) / spacing[0])
        voxelY = int((coordY - origin[1]) / spacing[1])
        voxelZ = int((coordZ - origin[2]) / spacing[2])
        # extraer patch
        patch = extract_patch(
            scan_norm,
            (voxelZ, voxelY, voxelX),
            patch_size=(64,64,64)
        )
        # verificar tamaño
        if patch.shape == (64,64,64):
            save_name = f"{seriesuid}_{idx}_label{label}.npy"
            save_path = os.path.join(patch_dir,save_name)
            # si ya existe lo salta
            if os.path.exists(save_path):continue
            # guardar patch
            np.save(save_path, patch)
            count += 1
            # imprimir cada 10000
            if count % 10000 == 0:
                print(f"Patches guardados: {count}")
    except Exception as e:

        print("Error:", e)

print("Total patches:", count)

Patches guardados: 10000
Patches guardados: 20000
Patches guardados: 30000
Patches guardados: 40000
Patches guardados: 50000
Patches guardados: 60000
Patches guardados: 70000
Patches guardados: 80000
Patches guardados: 90000
Patches guardados: 100000
Patches guardados: 110000
Patches guardados: 120000
Patches guardados: 130000
Patches guardados: 140000
Patches guardados: 150000
Patches guardados: 160000
Patches guardados: 170000
Patches guardados: 180000


KeyboardInterrupt: 

# Para extraer todos los que me faltan 

In [4]:
processed_idx = []
for file in os.listdir(patch_dir):
    if file.endswith(".npy"):
        try:
            idx = int(
                file.split("_label")[0].split("_")[-1]
            )
            processed_idx.append(idx)
        except:
            pass
remaining = candidates.drop(processed_idx)
remaining.to_csv(
    "remaining_candidates.csv",
    index=False
)
print("Procesados:", len(processed_idx))
print("Restantes:", len(remaining))

Procesados: 397277
Restantes: 153788


## Para borrar los scans procesados que ya utilice

In [6]:
remaining = pd.read_csv(
    "remaining_candidates.csv"
)
remaining_seriesuid = set(
    remaining['seriesuid'].unique()
)
for i in range(10):
    folder = f"processed_subset{i}"
    files = glob.glob(
        os.path.join(folder, "*.npy")
    )
    for file in files:
        seriesuid = os.path.basename(
            file
        ).replace(".npy", "")
        if seriesuid not in remaining_seriesuid:
            os.remove(file)
            print(f"Borrado: {seriesuid}")

Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674661221381920536987
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.146429221666426688999739595820
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.250863365157630276148828903732
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.281489753704424911132261151767
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.293757615532132808762625441831
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.313835996725364342034830119490
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.317087518531899043292346860596
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.114218724025049818743426522343
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.162207236104936931957809623059
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.247769845138587733933485039556
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.275766318636944297772360944907
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.325164338773720548739146851679
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.6001.336894364358709782463716339027
Borrado: 1.3.6.1.4.1.14519.5.2.1.6279.

## Para verificar las que faltan

In [26]:
total_processed = 0
for i in range(10):
    folder = f"processed_subset{i}"
    files = [
        f for f in os.listdir(folder)
        if f.endswith(".npy")
    ]
    cantidad = len(files)
    total_processed += cantidad
    print(f"{folder}: {cantidad}")
print("-" * 30)
print("TOTAL:", total_processed)

processed_subset0: 82
processed_subset1: 83
processed_subset2: 77
processed_subset3: 78
processed_subset4: 74
processed_subset5: 76
processed_subset6: 78
processed_subset7: 77
processed_subset8: 83
processed_subset9: 74
------------------------------
TOTAL: 782


In [24]:
positive = 0
negative = 0
for file in os.listdir(patch_dir):
    if file.endswith(".npy"):
        if "_label1.npy" in file:
            positive += 1
        elif "_label0.npy" in file:
            negative += 1
print("Positivos:", positive)
print("Negativos:", negative)
print("Total:", positive + negative)

Positivos: 1220
Negativos: 9688
Total: 10908


In [23]:
positive_remaining = 0
negative_remaining = 0
for label in remaining['class']:
    if label == 1:
        positive_remaining += 1
    elif label == 0:
        negative_remaining += 1
print("Positivos restantes:", positive_remaining)
print("Negativos restantes:", negative_remaining)
print(
    "Total restantes:",
    positive_remaining + negative_remaining
)

Positivos restantes: 131
Negativos restantes: 153428
Total restantes: 153559


# Para ver cuantas anotaciones hay por tomografia 

In [ ]:
#Le cambio a 1 para ver los positivos
positive_candidates = candidates[
    candidates['class'] == 0
]
count_per_scan = positive_candidates.groupby(
    'seriesuid'
).size()
print(count_per_scan)

seriesuid
1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860     689
1.3.6.1.4.1.14519.5.2.1.6279.6001.100332161840553388986847034053     490
1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793540579077826395208     958
1.3.6.1.4.1.14519.5.2.1.6279.6001.100530488926682752765845212286     466
1.3.6.1.4.1.14519.5.2.1.6279.6001.100620385482151095585000946543     432
                                                                    ... 
1.3.6.1.4.1.14519.5.2.1.6279.6001.979083010707182900091062408058     832
1.3.6.1.4.1.14519.5.2.1.6279.6001.980362852713685276785310240144     357
1.3.6.1.4.1.14519.5.2.1.6279.6001.986011151772797848993829243183    1106
1.3.6.1.4.1.14519.5.2.1.6279.6001.994459772950022352718462251777     173
1.3.6.1.4.1.14519.5.2.1.6279.6001.997611074084993415992563148335     738
Length: 888, dtype: int64


In [ ]:
count_per_scan = count_per_scan.sort_values(
    ascending=False
)
print(count_per_scan)

seriesuid
1.3.6.1.4.1.14519.5.2.1.6279.6001.339142594937666268384335506819    1467
1.3.6.1.4.1.14519.5.2.1.6279.6001.174692377730646477496286081479    1424
1.3.6.1.4.1.14519.5.2.1.6279.6001.279953669991076107785464313394    1414
1.3.6.1.4.1.14519.5.2.1.6279.6001.113697708991260454310623082679    1408
1.3.6.1.4.1.14519.5.2.1.6279.6001.258220324170977900491673635112    1398
                                                                    ... 
1.3.6.1.4.1.14519.5.2.1.6279.6001.608029415915051219877530734559     172
1.3.6.1.4.1.14519.5.2.1.6279.6001.397202838387416555106806022938      65
1.3.6.1.4.1.14519.5.2.1.6279.6001.225515255547637437801620523312      56
1.3.6.1.4.1.14519.5.2.1.6279.6001.935683764293840351008008793409      51
1.3.6.1.4.1.14519.5.2.1.6279.6001.153536305742006952753134773630      29
Length: 888, dtype: int64


# Teniendo en cuenta que esta desbalanceado me voy a quedar con 5 por cada tomografia

In [11]:
# agrupar negativos por tomografía
negative_groups = defaultdict(list)
for file in os.listdir(patch_dir):

    if "_label0.npy" in file:

        seriesuid = "_".join(
            file.split("_")[:-2]
        )

        negative_groups[seriesuid].append(file)
# máximo negativos por CT
max_negatives = 5
deleted = 0
for seriesuid, files in negative_groups.items():
    if len(files) > max_negatives:
        # escoger cuáles conservar
        keep = random.sample(
            files,
            max_negatives
        )
        # borrar resto
        for file in files:
            if file not in keep:
                os.remove(
                    os.path.join(
                        patch_dir,
                        file
                    )
                )
                deleted += 1
print("Negativos borrados:", deleted)

Negativos borrados: 4844


In [12]:
positive = 0
negative = 0
for file in os.listdir(patch_dir):
    if "_label1.npy" in file:
        positive += 1
    elif "_label0.npy" in file:
        negative += 1
print("Positivos:", positive)
print("Negativos:", negative)

Positivos: 1220
Negativos: 3460


In [19]:
# Para este tengo q ejecutar el de metadata,patch y la ruta creo q era
# SOLO POSITIVOS RESTANTES
remaining = pd.read_csv("remaining_candidates.csv")
remaining_pos = remaining[
    remaining['class'] == 1
].copy()
count = 0
for idx, row in remaining_pos.iterrows():
    seriesuid = row['seriesuid']
    coordX = row['coordX']
    coordY = row['coordY']
    coordZ = row['coordZ']
    label = row['class']
    # buscar processed scan
    norm_path = processed_paths.get(seriesuid)
    if not norm_path:
        continue
    try:
        # cargar scan preprocessado
        scan_norm = np.load(norm_path)
        # metadata cache
        metadata = metadata_cache.get(seriesuid)
        if metadata is None:
            continue
        origin = metadata["origin"]
        spacing = metadata["spacing"]
        # coordenadas reales -> voxel
        voxelX = int(
            (coordX - origin[0]) / spacing[0]
        )
        voxelY = int(
            (coordY - origin[1]) / spacing[1]
        )
        voxelZ = int(
            (coordZ - origin[2]) / spacing[2]
        )
        # extraer patch
        patch = extract_patch(
            scan_norm,
            (voxelZ, voxelY, voxelX),
            patch_size=(64,64,64)
        )
        # validar tamaño
        if patch.shape == (64,64,64):
            save_name = (
                f"{seriesuid}_{idx}_label{label}.npy"
            )
            save_path = os.path.join(
                patch_dir,
                save_name
            )
            # si ya existe lo salta
            if os.path.exists(save_path):
                continue
            # guardar patch
            np.save(save_path, patch)
            count += 1
            # BORRAR FILA DEL REMAINING
            remaining = remaining.drop(idx)
            # verificar si quedan filas
            still_exists = (
                remaining['seriesuid'] == seriesuid
            ).any()
            # si ya no quedan filas:
            # borrar processed scan
            if not still_exists:
                if os.path.exists(norm_path):
                    os.remove(norm_path)
                    print(
                        f"Processed borrado: "
                        f"{seriesuid}"
                    )
            # guardar remaining actualizado
            remaining.to_csv(
                "remaining_candidates.csv",
                index=False
            )
            # imprimir avance
            if count % 50 == 0:
                print(
                    f"Patches positivos: {count}"
                )
    except Exception as e:
        print("Error:", e)
print("Total positivos nuevos:", count)

Total positivos nuevos: 0


# Hacer el dataset

In [ ]:
#creo la lista
dataset = []
#Este recorre el candidates.csv
for file in os.listdir(patch_dir):
    if file.endswith(".npy"):
        # construye ruta
        path = os.path.join(patch_dir,file)
        # label
        if "_label1.npy" in file:
            label = 1
        elif "_label0.npy" in file:
            label = 0
        else:
            continue
        # seriesuid
        seriesuid = "_".join(
            file.split("_")[:-2]
        )
        dataset.append([path,label,seriesuid])
#Este lo convierte a tablita 
dataset_df = pd.DataFrame(
    dataset,
    columns=[
        "path",
        "label",
        "seriesuid"
    ]
)
#Con este lo mezclo paq no te juntito
dataset_df = dataset_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)
dataset_df.to_csv(
    "patch_dataset.csv",
    index=False
)

print(dataset_df.head())
print("Total:",len(dataset_df))
print(dataset_df['label'].value_counts())

                                                path  label  \
0  patches_candidates\1.3.6.1.4.1.14519.5.2.1.627...      0   
1  patches_candidates\1.3.6.1.4.1.14519.5.2.1.627...      1   
2  patches_candidates\1.3.6.1.4.1.14519.5.2.1.627...      0   
3  patches_candidates\1.3.6.1.4.1.14519.5.2.1.627...      0   
4  patches_candidates\1.3.6.1.4.1.14519.5.2.1.627...      1   

                                           seriesuid  
0  1.3.6.1.4.1.14519.5.2.1.6279.6001.306140003699...  
1  1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...  
2  1.3.6.1.4.1.14519.5.2.1.6279.6001.211071908915...  
3  1.3.6.1.4.1.14519.5.2.1.6279.6001.334166493392...  
4  1.3.6.1.4.1.14519.5.2.1.6279.6001.242624386080...  
Total: 4680
label
0    3460
1    1220
Name: count, dtype: int64


# Split

In [ ]:
#Obtener tomografia unica yaq una tiene muchos patches
unique_series = dataset_df['seriesuid'].unique()
#split train /temp 
train_series, temp_series = train_test_split(
    unique_series,
    test_size=0.30,
    random_state=42
)
#split validation / test
val_series, test_series = train_test_split(
    temp_series,
    test_size=0.50,
    random_state=42
)

In [23]:
#entreno
train_df = dataset_df[
    dataset_df['seriesuid']
    .isin(train_series)
]
#validacion
val_df = dataset_df[
    dataset_df['seriesuid']
    .isin(val_series)
]
#testeo
test_df = dataset_df[
    dataset_df['seriesuid']
    .isin(test_series)
]

In [7]:
#ver tamaños
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("-------------------------------------")
#ver balance
print(train_df['label'].value_counts())
print(val_df['label'].value_counts())
print(test_df['label'].value_counts())
#Guardar csv
train_df.to_csv(
    "train_dataset.csv",
    index=False
)
val_df.to_csv(
    "val_dataset.csv",
    index=False
)
test_df.to_csv(
    "test_dataset.csv",
    index=False
)

Train: 3307
Validation: 688
Test: 685
-------------------------------------
label
0    2445
1     862
Name: count, dtype: int64
label
0    500
1    188
Name: count, dtype: int64
label
0    515
1    170
Name: count, dtype: int64


# Verificación final

In [9]:
print("TRAIN")
print(
    train_df['label']
    .value_counts()
)

print("\nVALIDATION")
print(
    val_df['label']
    .value_counts()
)

print("\nTEST")
print(
    test_df['label']
    .value_counts()
)

TRAIN
label
0    2445
1     862
Name: count, dtype: int64

VALIDATION
label
0    500
1    188
Name: count, dtype: int64

TEST
label
0    515
1    170
Name: count, dtype: int64


# Overlap

In [17]:
train_series = set(
    train_df['seriesuid']
)

val_series = set(
    val_df['seriesuid']
)

test_series = set(
    test_df['seriesuid']
)

print(
    "Train-Val overlap:",
    len(train_series & val_series)
)

print(
    "Train-Test overlap:",
    len(train_series & test_series)
)

print(
    "Val-Test overlap:",
    len(val_series & test_series)
)

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


# Archivos que faltan

In [18]:
missing = 0
for path in dataset_df['path']:
    if not os.path.exists(path):
        missing += 1
print("Archivos faltantes:",
      missing)

Archivos faltantes: 0


# Verifica el tamaño

In [19]:
wrong_shape = 0
for path in dataset_df['path']:
    patch = np.load(path)
    if patch.shape != (64,64,64):
        wrong_shape += 1
print(
    "Patches con shape incorrecto:",
    wrong_shape
)

Patches con shape incorrecto: 0


In [20]:
nan_count = 0
for path in dataset_df['path']:
    patch = np.load(path)
    if np.isnan(patch).any():
        nan_count += 1
print("Patches con NaN:",
      nan_count)

Patches con NaN: 0
